In [2]:
import numpy as np
import matplotlib.pyplot as plt
import time
import cv2

## Hồng Sen – Demo 4: Nhiễu, lọc trung vị

In [ ]:
from skimage.util import random_noise
from skimage.metrics import mean_squared_error as calc_mse

# Tạo ảnh gốc kiểm thử
img_demo4 = np.zeros((256, 256), dtype=np.uint8)
img_demo4[64:192, 64:192] = 200

levels = [0.01, 0.05, 0.10, 0.15, 0.20, 0.30]
mse_med_list = []
mse_gauss_list = []

for val in levels:
    noisy_sp = random_noise(img_demo4, mode='s&p', amount=val)
    noisy_sp = (noisy_sp * 255).astype(np.uint8)
    
    filtered_median = cv2.medianBlur(noisy_sp, 3)
    filtered_gauss = cv2.GaussianBlur(noisy_sp, (3, 3), 0)
    
    mse_med = calc_mse(img_demo4, filtered_median)
    mse_gauss = calc_mse(img_demo4, filtered_gauss)
    mse_med_list.append(mse_med)
    mse_gauss_list.append(mse_gauss)

# Vẽ và lưu đồ thị độc lập
plt.figure(figsize=(8, 4))
plt.plot(levels, mse_med_list, marker='o', label='Median Filter', color='blue')
plt.plot(levels, mse_gauss_list, marker='s', label='Gaussian Filter', color='red')
plt.xlabel('Noise Level')
plt.ylabel('MSE Value')
plt.title('MSE Comparison')
plt.legend()
plt.grid(True)
plt.savefig('bieu_do_mse.png', dpi=300, bbox_inches='tight')
plt.close()

## Mở rộng: lọc song phương

In [ ]:
# Mở rộng: Lọc song phương (Bilateral Filter) giúp giữ biên cạnh
img_test = np.zeros((100, 100), dtype=np.uint8)
img_test[30:70, 30:70] = 250

noisy_bilateral = cv2.GaussianBlur(img_test, (5, 5), 2)
filtered_bilateral = cv2.bilateralFilter(noisy_bilateral, d=9, sigmaColor=75, sigmaSpace=75)

## KT2 – Ảnh tích phân: hàm

In [ ]:
def integral_image(img):
    return cv2.integral(img.astype(np.float32))

def adaptive_threshold_integral(img, s=15, t=15):
    integ = integral_image(img)
    H, W = img.shape
    S = s // 2
    out = np.zeros_like(img)
    
    for r in range(H):
        for c in range(W):
            x1 = max(0, c - S)
            x2 = min(W - 1, c + S)
            y1 = max(0, r - S)
            y2 = min(H - 1, r + S)
            
            count = (x2 - x1 + 1) * (y2 - y1 + 1)
            total = integ[y2+1, x2+1] - integ[y1, x2+1] - integ[y2+1, x1] + integ[y1, x1]
            
            if int(img[r, c]) * count < total * (1.0 - t/100.0):
                out[r, c] = 0
            else:
                out[r, c] = 255
    return out

## KT2 – Thí nghiệm

In [ ]:
test_paper = np.ones((100, 100), dtype=np.uint8) * 200
test_paper[40:60, 40:60] = 50

out_thresh = adaptive_threshold_integral(test_paper)